# Description

Number density of points in a sine wave-like curve

# Load analysis functions

In [1]:
%run ....../imports.ipynb
%run ...../snotelAnalysisFunctions.ipynb

# Station Example

## Berthoud Pass

In [2]:
state = 'CO'
chosenStation = 'Berthoud Summit (335)'
year = 'all' # can be a single year, or "all"

In [3]:
stationBiasDict = loadTempBiasMarker()

In [4]:
rootPath = '...../snotelData/raw/'
stateNames = [state]#,'AZ','CA','CO','ID','MT','NM','NV','OR','SD','UT','WA','WY']
metaData = loadStationMetaData()

In [6]:
allStateDict = {}
for stateName in tqdm(stateNames):
    stateFileNames = glob.glob(rootPath + stateName + '/' + '*')
    stateDataAll = [csvToXarray(f) for f in stateFileNames]
    stateDataCleaned = [s for s in stateDataAll if s.BAD_DATA==False]
    allStateDict[stateName] = stateDataCleaned

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.92s/it]


In [7]:
allStateDict = filterStations(stateNames,allStateDict)

['CO']: 116/117         

In [8]:
currStation = None
for s in allStateDict[state]:
    if s.name == chosenStation:
        currStation = s

In [9]:
# Check if temperature bias has been corrected
stationBiasBoolDefault = True
try:
    stationBiasBool = stationBiasDict[currStation.number]=='Complete'
except KeyError:
    stationBiasBool = stationBiasBoolDefault
    #print('No bias data')
print(stationBiasBool)

True


Select temperatures, both all year and in just winter

In [17]:
winterInds = np.logical_or(currStation.time.dt.month>=11,currStation.time.dt.month<=3)
winterTemps = currStation.T_avg[winterInds]

compInds = np.logical_and(currStation.time.dt.month>=4,currStation.time.dt.month<=10)
compTemps = currStation.T_avg[compInds]

allTemps = currStation.T_avg

In [18]:
## Save for final plot ##
saveBool = True
if saveBool:
    pathToSave = '..../figureData'
    np.savez(pathToSave + '/figure7Dist.npz', 
             winterInds=winterInds,
             winterTemps = winterTemps,
             compTemps = compTemps,
             allTemps = allTemps)